In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import string
from sklearn.metrics import classification_report
import tensorflow as tf
from keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping,ModelCheckpoint
from tensorflow.keras.layers import Dense, Input, Dropout, Bidirectional, LSTM, Embedding, BatchNormalization,  Reshape, Conv2D, MaxPool2D, concatenate, Flatten, Activation
import torch
import numpy as np
from transformers import BertTokenizer, BertModel,RobertaTokenizer, RobertaModel,AutoTokenizer, AutoModel
import ast

In [ ]:
# Verificar si la GPU está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
archivo_3 = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/ResultsBERT/DataProgramsandDescriptions-CatRetroalimentacion5000.xlsx'
train_full = pd.read_excel(archivo_3)

# AST spliter D. Gries form

In [ ]:
class WhileLoopFinder(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code.splitlines()
        self.functions_with_while = []
        self.current_function = None

    def visit_FunctionDef(self, node):
        original_current_function = self.current_function
        self.current_function = {
            "function_name": node.name,
            "has_while_loop": False,
            "pre_while_code_lines": [],
            "while_loops": []
        }

        function_start_line = node.lineno - 1
        function_end_line = (node.end_lineno if hasattr(node, 'end_lineno') else len(self.source_code))
        function_lines = self.source_code[function_start_line:function_end_line]

        found_while = False
        for stmt in node.body:
            if isinstance(stmt, ast.While):
                self.current_function["has_while_loop"] = True
                found_while = True
                try:
                    while_condition = ast.unparse(stmt.test).strip()
                    print("OK condition")
                except AttributeError:
                    print("Error al obtener la condición del while")
                try:
                    while_body_code = ast.unparse(ast.Module(body=stmt.body, type_ignores=[])).strip()
                    print("Ok body")
                except AttributeError:
                    print("Error al obtener el código del cuerpo del while")

                self.current_function["while_loops"].append({
                    "condition": while_condition,
                    "body_code": while_body_code
                })
            elif not found_while:
                start_line = stmt.lineno - 1
                end_line = (stmt.end_lineno if hasattr(stmt, 'end_lineno') else stmt.lineno)
                self.current_function["pre_while_code_lines"].extend(self.source_code[start_line:end_line])
            self.generic_visit(stmt)

        if self.current_function["has_while_loop"]:
            last_while_end_line = None
            if self.current_function["while_loops"]:
                last_while_node = next((node for node in reversed(node.body) if isinstance(node, ast.While)), None)
                if last_while_node and hasattr(last_while_node, 'end_lineno'):
                    last_while_end_line = last_while_node.end_lineno
            post_while_code_lines = []
            if last_while_end_line:
                function_indent = len(self.source_code[node.lineno - 1]) - len(self.source_code[node.lineno - 1].lstrip())
                for line in self.source_code[last_while_end_line:function_end_line]:
                    if line.startswith(self.source_code[node.lineno - 1][:function_indent] + "    "):
                        post_while_code_lines.append(line[function_indent + 4:])
                    else:
                        post_while_code_lines.append(line[function_indent:])
            self.current_function["post_while_code_lines"] = "\n".join(post_while_code_lines).strip()
            self.current_function["pre_while_code_lines"] = "\n".join(self.current_function["pre_while_code_lines"]).strip()
            self.functions_with_while.append(self.current_function)

        self.current_function = original_current_function



In [ ]:
def DGries_states(solution):
  try:
    # Crear el AST
    tree = ast.parse(solution)
    # Recorrer el AST con nuestro visitante
    finder = WhileLoopFinder(solution)
    finder.visit(tree)
    for func_info in finder.functions_with_while:
      initial_state=func_info["pre_while_code_lines"] if func_info["pre_while_code_lines"] else False
      end_state=""
      transformation_state=""
      for j, wl in enumerate(func_info["while_loops"]):
          end_state=wl["condition"]
          transformation_state=wl["body_code"]

      if not initial_state:
        initial_state=transformation_state
      return initial_state,transformation_state,end_state
  except Exception as error:
    return "Exception:"+str(error),"Exception:"+str(error),"Exception:"+str(error)




# Encoder Description and Code

In [ ]:
class Encoder:
  def __init__(self):
    self.is_loadtokenizers=False



  def tokenize_and_generate_embeddings_descriptions(self,description):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input description and move tokens to GPU
    tokens = self.beart_tokenizer.encode_plus(description, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.beart_model(**tokens)
    # Extract embeddings for all tokens
    desc_embeddings = outputs.last_hidden_state.cpu().numpy()
    return desc_embeddings

  def tokenize_and_generate_embeddings_codes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.codebeart_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.codebeart_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()

    return code_embeddings


  def tokenize_and_generate_embeddings_graphcodes(self,code):
    if not self.is_loadtokenizers:
      raise ValueError("tokenizers dont load")
    # Tokenize input code and move tokens to GPU
    tokens = self.graphcodebert_tokenizer.encode_plus(code, return_tensors="pt", truncation=True)
    tokens = {key: value.to(device) for key, value in tokens.items()}
    # Generate embeddings
    with torch.no_grad():
        outputs = self.graphcodebert_model(**tokens)
    # Extract embeddings for all tokens
    code_embeddings = outputs.last_hidden_state.cpu().numpy()
    return code_embeddings

  def load_beart_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    model_name = "bert-base-uncased"
    self.beart_model = BertModel.from_pretrained(model_name)
    self.beart_tokenizer = BertTokenizer.from_pretrained(model_name)
    self.beart_model.to(device)

  def load_graphcodebert_tokenizer(self):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.graphcodebert_tokenizer = AutoTokenizer.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model = AutoModel.from_pretrained("microsoft/graphcodebert-base")
    self.graphcodebert_model.to(device)


  def load_codebert_tokenizer(self):
    # Verificar si la GPU está disponible
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Cargar el tokenizer y el modelo en la GPU si está disponible
    self.codebeart_tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    self.codebeart_model = RobertaModel.from_pretrained("microsoft/codebert-base")
    self.codebeart_model.to(device)

  def start_tokenizer(self):
    self.load_beart_tokenizer()
    #self.load_codebert_tokenizer()
    self.load_graphcodebert_tokenizer()
    self.is_loadtokenizers=True




# All characteristics

In [ ]:
def categoricallabelAll(w):
  if w=="['Correct']":
    return 0
  if w=="['Initial state']":
    return 1
  if w=="['Final state']":
    return 2
  if w=="['State transformation']":
    return 3
  if w=="['Initial state', 'Final state']":
    return 4
  if w=="['Initial state', 'State transformation']":
    return 5
  if w=="['Final state', 'State transformation']":
    return 6
  if w=="['Initial state', 'Final state', 'State transformation']":
    return 7
  return 8

category=np.array([
    'Correct',
    'Initial state',
    'Final state',
    'State transformation',
    'Initial state, Final state',
    'Initial state, State transformation',
    'Final state, State transformation',
    'Initial state, Final state, State transformation'

])

# Load Dataset

In [ ]:
train_full.head()

,No.,Problema,Solución,Estado incial,Estado final,Transformación de estado,Etiqueta 1,Etiqueta 2,Realimentación
0,1,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,"result = 1, i = 1",i <= n,"result *= i, i += 1",Correct,['Correct'],NaN
1,2,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,"total = 0, i = 1, n = 100",i <= n,"total += i, i += 1",Correct,['Correct'],NaN
2,3,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,"numbers = [], i = 0, n = 10",i <= n,"print(i), numbers.append(i), i += 1",Correct,['Correct'],NaN
3,4,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,"numbers = [], i = 10, n = 1",i >= n,"print(i), numbers.append(i), i -= 1",Correct,['Correct'],NaN
4,5,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,"i = 2, i = = 0:",i <= num//2,"if num % i == 0:, return False, i += 1",Correct,['Correct'],NaN


In [ ]:
train_full.drop(["No.","Realimentación","Estado incial","Estado final","Transformación de estado"],axis=1,inplace=True)
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct']
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct']
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct']
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct']
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct']
...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']"
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']"
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']"


In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Solución'].apply(DGries_states).apply(pd.Series)

Streaming output truncated to the last 5000 lines.
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK c

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [ ]:
train_full.dropna(inplace=True)

In [ ]:
np.unique(train_full['Etiqueta 2'])

array(["['Correct']", "['Final state', 'State transformation']",
       "['Final state']",
       "['Initial state', 'Final state', 'State transformation']",
       "['Initial state', 'Final state']",
       "['Initial state', 'State transformation']", "['Initial state']",
       "['State transformation']"], dtype=object)

In [ ]:
train_full

,Problema,Solución,Etiqueta 1,Etiqueta 2,Estado incial,Transformación de estado,Estado final
0,Write a Python function that returns the facto...,def factorial(n):\n result = 1\n i = 1\n...,Correct,['Correct'],result = 1\n i = 1,result *= i\ni += 1,i <= n
1,Write a Python function that returns the sum o...,def sum_1_to_100():\n total = 0\n i = 1\...,Correct,['Correct'],total = 0\n i = 1\n n = 100,total += i\ni += 1,i <= n
2,Write a Python function that prints the number...,def print_and_store():\n numbers = []\n ...,Correct,['Correct'],numbers = []\n i = 0\n n = 10,print(i)\nnumbers.append(i)\ni += 1,i <= n
3,Write a Python function to print the numbers f...,def print_and_store_reverse():\n numbers = ...,Correct,['Correct'],numbers = []\n i = 10\n n = 1,print(i)\nnumbers.append(i)\ni -= 1,i >= n
4,Write a Python function to check if a number i...,def is_prime(num):\n if num <= 1:\n ...,Correct,['Correct'],if num <= 1:\n return False\n i = 2,if num % i == 0:\n return False\ni += 1,i <= num // 2
...,...,...,...,...,...,...,...
4995,Write a Python function that returns the sum o...,def sum_of_first_five_numbers():\n number =...,Incorrect,"['Initial state', 'Final state']",number = 1\n total_sum = 0\n count = 1,total_sum = total_sum + number\ncount = count - 1,count < 5
4996,Write a Python function that returns the facto...,def factorial(n):\n result = 0\n i = 0\n...,Incorrect,"['Initial state', 'Final state']",result = 0\n i = 0,result *= i\ni += 1,i < n
4997,Write a Python function that prints the number...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']",numbers = []\n n = 1,print(n)\nnumbers.append(n)\nn += 1,n <= 11
4998,Write a Python function to print the numbers f...,def print_and_store_numbers():\n numbers = ...,Incorrect,"['Initial state', 'Final state']",numbers = []\n n = 9,print(n)\nnumbers.append(n)\nn -= 1,n >= 0


In [ ]:
encoder=Encoder()
encoder.start_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
%%time
problem=train_full['Problema'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Solución'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 3min 10s, sys: 816 ms, total: 3min 10s
Wall time: 3min 10s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((4919,), (4919,), (4919,), (4919,), (4919,))

In [ ]:
print(startstate.shape)
print(startstate[1262].shape)


(4919,)
(1, 17, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((4919, 768), (4919, 768), (4919, 768), (4919, 768))

In [ ]:
y=train_full['Etiqueta 2'].apply(categoricallabelAll)
y=y.to_numpy()
np.unique(y)

array([0, 1, 2, 3, 4, 5, 6, 7])

In [ ]:
from sklearn.model_selection import train_test_split

Xp_train,Xp_test,Xcode_train,Xcode_test,Xs_train,Xs_test,Xt_train,Xt_test,Xf_train,Xf_test,y_train,y_test=train_test_split(Xp,Xcode,Xs,Xt,Xf,y,test_size=0.2,random_state=2023, stratify=y)

In [ ]:
Xp_train.shape,Xp_test.shape,Xcode_train.shape, Xcode_test.shape, Xs_train.shape,Xs_test.shape,Xt_train.shape,Xt_test.shape,Xf_train.shape,Xf_test.shape,y_train.shape,y_test.shape

((3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935, 768),
 (984, 768),
 (3935,),
 (984,))

In [ ]:
#random search
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import matthews_corrcoef
#XGBoost
from xgboost import XGBClassifier
#pipeline
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier


In [ ]:
from sklearn.metrics import matthews_corrcoef, precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def calculate_mcc_multiclass(y_true, y_pred_probs):
    y_pred_labels = np.argmax(y_pred_probs, axis=1)
    # Convertir etiquetas verdaderas one-hot a etiquetas enteras si es necesario
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    return matthews_corrcoef(y_true, y_pred_labels)

def calculate_auc_pr_multiclass(y_true, y_pred_probs, average='macro'):
    n_classes = y_pred_probs.shape[1]
    #y_true_labels = np.argmax(y_true, axis=1) if y_true.ndim > 1 else y_true
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    auc_pr_list = []
    for i in range(n_classes):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs[:, i])
        auc_pr_list.append(auc(recall, precision))

    if average == 'macro':
        return np.mean(auc_pr_list)
    elif average == 'weighted':
        class_counts = y_true_bin.sum(axis=0)
        return np.average(auc_pr_list, weights=class_counts)
    else:
        return auc_pr_list


# KNN

In [ ]:
from tensorflow.keras.losses import sparse_categorical_crossentropy
from sklearn.ensemble import VotingClassifier,StackingClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
accuracies_train=[]
accuracies_predict=[]
loss_predict=[]
times_train=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]
ml_names=[]
abl_names=[]

In [ ]:
import joblib

In [ ]:
path="/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/1. Tradicional Machine Learning/"

In [ ]:
from time import time
knn=KNeighborsClassifier(n_neighbors=3,p=1,weights='distance')
svm_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',SVC(kernel='rbf',gamma=0.000379269019073225,C=233.57214690901213,probability=True))
])
mlp_pipe=Pipeline([
    ('scaler',StandardScaler()),
    ('classifier',MLPClassifier(hidden_layer_sizes=(100,100,100),learning_rate='adaptive'))
])

rf=RandomForestClassifier(n_estimators=70,max_depth=15)

vt=VotingClassifier(estimators=[('mlp',mlp_pipe),('rf',rf)],voting='soft')
st=StackingClassifier(estimators=[('mlp',mlp_pipe),('rf',rf)],final_estimator=LogisticRegression())

models=[knn,svm_pipe,mlp_pipe,rf,vt,st]
models_names=["KNN","SVM","MLP","Random Forest","Voting","Stacking"]
ablations_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

for model,name in zip(models,models_names):
  print(name)
  # Define the full datasets outside the inner loop
  X_train_full = np.concatenate((Xp_train,Xcode_train,Xs_train,Xt_train,Xf_train),axis=1)
  X_test_full = np.concatenate((Xp_test,Xcode_test,Xs_test,Xt_test,Xf_test),axis=1)

  for ablations_name in ablations_names:
    ml_names.append(name)
    abl_names.append(ablations_name)
    print(ablations_name)

    if ablations_name=="problemall":
      X_train = X_train_full
      X_test = X_test_full
    elif ablations_name=="problem_gries":
      X_train=np.concatenate((Xp_train,Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xp_test,Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="code_gries":
      X_train=np.concatenate((Xcode_train,Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xcode_test,Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="gries":
      X_train=np.concatenate((Xs_train,Xt_train,Xf_train),axis=1)
      X_test=np.concatenate((Xs_test,Xt_test,Xf_test),axis=1) # Corrected X_test construction
    elif ablations_name=="problem_code":
      X_train=np.concatenate((Xp_train,Xcode_train),axis=1)
      X_test=np.concatenate((Xp_test,Xcode_test),axis=1) # Corrected X_test construction
    elif ablations_name=="code":
      X_train=Xcode_train # Corrected X_train construction
      X_test=Xcode_test # Corrected X_test construction

    time1=time()
    model.fit(X_train,y_train)
    time2=time()
    joblib.dump(model,path+"graph_"+name+"_"+ablations_name+".joblib")
    times_train.append(time2-time1)
    time1=time()
    y_pred=model.predict(X_test)
    time2=time()
    times_predict.append(time2-time1)

    accuracie_train=model.score(X_train,y_train)
    accuracies_train.append(accuracie_train)
    accuracie_test=model.score(X_test,y_test)
    accuracies_predict.append(accuracie_test)
    #sparse_categorical_crossentropy
    y_pred_proba = model.predict_proba(X_test)
    loss=np.mean(sparse_categorical_crossentropy(y_test,y_pred_proba))
    loss_predict.append(loss)
    mcc = matthews_corrcoef(y_test, y_pred)
    mccs_predict.append(mcc)
    auc_pr = calculate_auc_pr_multiclass(y_test, model.predict_proba(X_test))
    aucpr_predict.append(auc_pr)
    print(classification_report(y_test,y_pred))
    print("Accuracy train",accuracie_train)
    print("Accuracy predict",accuracie_test)
    print("Loss",loss)
    print("MCC",mcc)
    print("AUC",auc_pr)
    print("==================================")

KNN
problemall
              precision    recall  f1-score   support

           0       0.90      0.83      0.86       292
           1       0.93      0.93      0.93       198
           2       0.94      0.95      0.95       197
           3       0.83      0.91      0.87       197
           4       0.93      0.96      0.94        26
           5       0.79      0.76      0.78        25
           6       1.00      0.93      0.96        28
           7       1.00      1.00      1.00        21

    accuracy                           0.90       984
   macro avg       0.91      0.91      0.91       984
weighted avg       0.90      0.90      0.90       984

Accuracy train 0.9956797966963151
Accuracy predict 0.9004065040650406
Loss 0.9900802896212624
MCC 0.8745241037893227
AUC 0.9515437658415379
problem_gries
              precision    recall  f1-score   support

           0       0.88      0.83      0.85       292
           1       0.93      0.94      0.94       198
           2     

In [ ]:
import pandas as pd
df=pd.DataFrame({
    "Model":ml_names,
    "Ablation":abl_names,
    "Accuracy_train":accuracies_train,
    "Accuracy_predict":accuracies_predict,
    "Loss":loss_predict,
    "Time_train":times_train,
    "Time_predict":times_predict,
    "MCC":mccs_predict,
    "AUC":aucpr_predict
})





In [ ]:
df

,Model,Ablation,Accuracy_train,Accuracy_predict,Loss,Time_train,Time_predict,MCC,AUC
0,KNN,problemall,0.995680,0.900407,0.990080,0.006368,5.087431,0.874524,0.951544
1,KNN,problem_gries,0.992122,0.894309,1.061069,0.004771,3.989551,0.866496,0.944327
2,KNN,code_gries,0.995426,0.900407,0.959353,0.004988,4.023134,0.874524,0.951783
3,KNN,gries,0.991105,0.896341,0.999062,0.004053,3.061593,0.868907,0.945153
4,KNN,problem_code,0.995172,0.899390,1.087104,0.003142,1.995194,0.873741,0.943620
5,KNN,code,0.995426,0.901423,1.040223,0.001751,0.999232,0.876112,0.948283
6,SVM,problemall,0.995172,0.939024,0.263309,134.778715,7.578310,0.922671,0.958902
7,SVM,problem_gries,0.991105,0.925813,0.294671,86.281680,6.228316,0.905763,0.956484
8,SVM,code_gries,0.994917,0.943089,0.256628,86.819225,6.569305,0.927824,0.960433
9,SVM,gries,0.989327,0.923780,0.293568,52.342694,4.118089,0.903145,0.957890


In [ ]:
path="/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/1. Tradicional Machine Learning/"

In [ ]:
df.to_csv(path+"graphtradicionalML_ablationsResults.csv",index=False)

In [ ]:
df=pd.read_csv(path+"graphtradicionalML_ablationsResults.csv")
df

,Model,Ablation,Accuracy_train,Accuracy_predict,Loss,Time_train,Time_predict,MCC,AUC
0,KNN,problemall,0.995680,0.900407,0.990080,0.006368,5.087431,0.874524,0.951544
1,KNN,problem_gries,0.992122,0.894309,1.061069,0.004771,3.989551,0.866496,0.944327
2,KNN,code_gries,0.995426,0.900407,0.959353,0.004988,4.023134,0.874524,0.951783
3,KNN,gries,0.991105,0.896341,0.999062,0.004053,3.061593,0.868907,0.945153
4,KNN,problem_code,0.995172,0.899390,1.087104,0.003142,1.995194,0.873741,0.943620
5,KNN,code,0.995426,0.901423,1.040223,0.001751,0.999232,0.876112,0.948283
6,SVM,problemall,0.995172,0.939024,0.263309,134.778715,7.578310,0.922671,0.958902
7,SVM,problem_gries,0.991105,0.925813,0.294671,86.281680,6.228316,0.905763,0.956484
8,SVM,code_gries,0.994917,0.943089,0.256628,86.819225,6.569305,0.927824,0.960433
9,SVM,gries,0.989327,0.923780,0.293568,52.342694,4.118089,0.903145,0.957890


# Externd Data

In [ ]:
extern_file = '/content/drive/MyDrive/UDENAR/ProyectosGrado/Errol/Ginna/Tuning/BestModels/Copia de ExternData.csv'
train_full = pd.read_csv(extern_file)
train_full

,Problem Description,Python Code,Source,Description Error,Error Label
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2


In [ ]:
train_full['Python Code']=train_full['Python Code'].apply(lambda x: x.replace("\\n","\n"))

In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [ ]:
train_full

,Problem Description,Python Code,Source,Description Error,Error Label,Estado incial,Transformación de estado,Estado final
0,Count how many even numbers are up to n,def count_evens_buggy(n):\n i = 0\n coun...,Inspired by CodeChef Learn Python (While Loop)...,missing increment leads to infinite loop,2,i = 0\n count = 0,if i % 2 == 0:\n count += 1,i <= n
1,Sum numbers from 1 to n,def sum_to_n_buggy(n):\n total = 0\n i =...,Based on typical mistakes in Codeforces common...,i is incorrectly incremented,2,total = 0\n i = 1,total += i\ni = i + 0,i <= n
2,Prompt user for numbers until 0 is entered,def input_until_zero_buggy():\n num = int(i...,Common error from GeeksforGeeks Python while l...,number is never updated in the loop,2,num = int(input('Enter number (0 to exit): ')),"print('Number:', num)",num != 0
3,Find first multiple of 7 greater than n,def first_multiple_7_buggy(n):\n i = n + 1\...,Typical error in CodeNet beginner loop problems,i not incremented,2,i = n + 1,pass,i % 7 != 0
4,Calculate factorial of n,def factorial_buggy(n):\n result = 1\n w...,Inspired by classic factorial exercises,"decrement by 2, not 1",2,result = 1,result *= n\nn -= 2,n > 1
5,Count digits in a positive integer n,def count_digits_buggy(n):\n count = 0\n ...,Inspired by digit counting problems in Codefor...,n not updated inside the loop,2,count = 0,n % 10\ncount += 1,n > 0
6,Sum all even numbers up to n,def sum_evens_buggy(n):\n total = 0\n i ...,"CodeChef/Codeforces summing problems, conditio...",loop condition incorrect,1,total = 0\n i = 2,total += i\ni += 2,i < n
7,Find a prime number using break,def find_prime_buggy():\n num = 2\n whil...,Inspired by break/continue loop errors from Py...,break outside the if block,2,num = 2,is_prime = True\ndivisor = 2\nwhile divisor < ...,True
8,Print cubes of numbers from 1 to n,def print_cubes_buggy(n):\n while i <= n:\n...,"Common initialization error, based on CodeNet ...",i not initialized,0,print(i ** 3)\ni += 1,print(i ** 3)\ni += 1,i <= n
9,Prompt for password until correct,def prompt_password_buggy():\n password = i...,Typical validation loop error from Python begi...,password requested only once,2,password = input('Enter password: '),print('Incorrect'),password != 'secret'


In [ ]:
train_full['Python Code']=train_full['Python Code'].apply(lambda x: x.replace("\\n","\n"))

In [ ]:
train_full[["Estado incial","Transformación de estado","Estado final"]]=train_full['Python Code'].apply(DGries_states).apply(pd.Series)

OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body
OK condition
Ok body


In [ ]:
%%time
problem=train_full['Problem Description'].apply(encoder.tokenize_and_generate_embeddings_descriptions).to_numpy()
code=train_full['Python Code'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
startstate = train_full['Estado incial'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
finalstate = train_full['Estado final'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()
transstate = train_full['Transformación de estado'].apply(encoder.tokenize_and_generate_embeddings_graphcodes).to_numpy()

CPU times: user 1.44 s, sys: 3 µs, total: 1.44 s
Wall time: 1.44 s


In [ ]:
problem.shape,code.shape,startstate.shape,finalstate.shape,transstate.shape

((35,), (35,), (35,), (35,), (35,))

In [ ]:
print(startstate.shape)
print(startstate[0].shape)


(35,)
(1, 12, 768)


In [ ]:
Xp=np.array([sentence[0].mean(axis=0) for sentence in problem])
Xcode=np.array([sentence[0].mean(axis=0) for sentence in code])
Xs=np.array([sentence[0].mean(axis=0) for sentence in startstate])
Xf=np.array([sentence[0].mean(axis=0) for sentence in finalstate])
Xt=np.array([sentence[0].mean(axis=0) for sentence in transstate])
Xp.shape,Xs.shape,Xf.shape,Xt.shape

((35, 768), (35, 768), (35, 768), (35, 768))

In [ ]:
y=train_full['Error Label']+1
y=y.to_numpy()
np.unique(y)

array([1, 2, 3, 4, 5, 6, 7])

In [ ]:
accuracies_predict=[]
loss_predict=[]
times_predict=[]
mccs_predict=[]
aucpr_predict=[]
ml_names=[]
abl_names=[]

ml_names=[]
abl_names=[]

In [ ]:
from time import time


models_names=["KNN","SVM","MLP","Random Forest","Voting","Stacking"]
ablations_names=["problemall","problem_gries","code_gries","gries","problem_code","code"]

for name in models_names:
  print(name)
  for ablations_name in ablations_names:
    ml_names.append(name)
    abl_names.append(ablations_name)
    print(ablations_name)
    model=joblib.load(path+"graph_"+name+"_"+ablations_name+".joblib")
    print(model)
    if ablations_name=="problemall":
      Xe= np.concatenate((Xp,Xcode,Xs,Xt,Xf),axis=1)
    elif ablations_name=="problem_gries":
      Xe=np.concatenate((Xp,Xs,Xt,Xf),axis=1)
    elif ablations_name=="code_gries":
      Xe=np.concatenate((Xcode,Xs,Xt,Xf),axis=1)
    elif ablations_name=="gries":
      Xe=np.concatenate((Xs,Xt,Xf),axis=1)
    elif ablations_name=="problem_code":
      Xe=np.concatenate((Xp,Xcode),axis=1)
    elif ablations_name=="code":
      Xe=Xcode

    time1=time()
    y_pred=model.predict(Xe)
    time2=time()
    times_predict.append(time2-time1)

    accuracie_test=model.score(Xe,y)
    accuracies_predict.append(accuracie_test)
    #sparse_categorical_crossentropy
    y_pred_proba = model.predict_proba(Xe)
    loss=np.mean(sparse_categorical_crossentropy(y,y_pred_proba))
    loss_predict.append(loss)
    mcc = matthews_corrcoef(y, y_pred)
    mccs_predict.append(mcc)
    auc_pr = calculate_auc_pr_multiclass(y, y_pred_proba)
    aucpr_predict.append(auc_pr)
    print(classification_report(y,y_pred))
    print("Accuracy predict",accuracie_test)
    print("Loss",loss)
    print("MCC",mcc)
    print("AUC",auc_pr)
    print("==================================")

KNN
problemall
KNeighborsClassifier(n_neighbors=3, p=1, weights='distance')


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.20      0.20      0.20         5
           3       0.75      0.12      0.21        24
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1

    accuracy                           0.11        35
   macro avg       0.12      0.04      0.05        35
weighted avg       0.54      0.11      0.18        35

Accuracy predict 0.11428571428571428
Loss 12.583647273350799
MCC 0.007026575862457513
AUC 0.2609902033730158
problem_gries
KNeighborsClassifier(n_neighbors=3, p=1, weights='distance')


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.20      0.20      0.20         5
           3       0.83      0.21      0.33        24
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1

    accuracy                           0.17        35
   macro avg       0.13      0.05      0.07        35
weighted avg       0.60      0.17      0.26        35

Accuracy predict 0.17142857142857143
Loss 12.531909572004695
MCC 0.032708842027137225
AUC 0.329060226521164
code_gries
KNeighborsClassifier(n_neighbors=3, p=1, weights='distance')


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.40      0.40      0.40         5
           3       0.86      0.25      0.39        24
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1

    accuracy                           0.23        35
   macro avg       0.16      0.08      0.10        35
weighted avg       0.64      0.23      0.32        35

Accuracy predict 0.22857142857142856
Loss 2.2551949861870084
MCC 0.09881335454324008
AUC 0.28532675009841
gries
StackingClassifier(estimators=[('mlp',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                 MLPClass

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         2
           2       0.50      0.40      0.44         5
           3       0.80      0.17      0.28        24
           4       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         1

    accuracy                           0.17        35
   macro avg       0.16      0.07      0.09        35
weighted avg       0.62      0.17      0.25        35

Accuracy predict 0.17142857142857143
Loss 2.2546924463112847
MCC 0.07962012158517946
AUC 0.2623595343481502
problem_code
StackingClassifier(estimators=[('mlp',
                                Pipeline(steps=[('scaler', StandardScaler()),
                                                ('classifier',
                                                

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

In [ ]:
import pandas as pd
dfe=pd.DataFrame({
    "Model":ml_names,
    "Ablation":abl_names,
    "Accuracy_predict":accuracies_predict,
    "Loss":loss_predict,
    "Time_predict":times_predict,
    "MCC":mccs_predict,
    "AUC":aucpr_predict
})





In [ ]:
dfe

,Model,Ablation,Accuracy_predict,Loss,Time_predict,MCC,AUC
0,KNN,problemall,0.114286,12.583647,0.205471,0.007027,0.260990
1,KNN,problem_gries,0.171429,12.531910,0.151275,0.032709,0.329060
2,KNN,code_gries,0.142857,12.565569,0.145434,0.058923,0.327121
3,KNN,gries,0.142857,12.553745,0.114011,0.024007,0.330506
4,KNN,problem_code,0.085714,13.472439,0.084205,-0.026806,0.300775
5,KNN,code,0.142857,11.705482,0.042005,0.059979,0.299440
6,SVM,problemall,0.257143,1.791666,0.293558,0.228446,0.286014
7,SVM,problem_gries,0.285714,1.670430,0.226819,0.269001,0.298341
8,SVM,code_gries,0.257143,1.889034,0.212783,0.052392,0.245884
9,SVM,gries,0.228571,2.098319,0.153664,0.103989,0.249370


In [ ]:
dfe.to_csv(path+"graph_externData_ablationsResults.csv",index=False)

In [ ]:
dfe=pd.read_csv(path+"graph_externData_ablationsResults.csv")
dfe

,Model,Ablation,Accuracy_predict,Loss,Time_predict,MCC,AUC
0,KNN,problemall,0.114286,12.583647,0.205471,0.007027,0.260990
1,KNN,problem_gries,0.171429,12.531910,0.151275,0.032709,0.329060
2,KNN,code_gries,0.142857,12.565569,0.145434,0.058923,0.327121
3,KNN,gries,0.142857,12.553745,0.114011,0.024007,0.330506
4,KNN,problem_code,0.085714,13.472439,0.084205,-0.026806,0.300775
5,KNN,code,0.142857,11.705482,0.042005,0.059979,0.299440
6,SVM,problemall,0.257143,1.791666,0.293558,0.228446,0.286014
7,SVM,problem_gries,0.285714,1.670430,0.226819,0.269001,0.298341
8,SVM,code_gries,0.257143,1.889034,0.212783,0.052392,0.245884
9,SVM,gries,0.228571,2.098319,0.153664,0.103989,0.249370
